# Supervised learning setup

_Machine Learning_

**Show the computer examples with answers. It learns to answer new ones.**

Supervised learning means learning from examples that already have answers.

     You collect many pairs: an input, and its correct output.

---

This notebook walks through the supervised learning loop one step at a time. Run each cell, read the note above it, and see how a model learns a rule from labeled pairs and then answers brand-new inputs. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

The supervised recipe has three moves: gather labeled examples, fit a model on a *training* split, then test it on inputs it never saw. We build it in three steps below.

### Step 1 — Make labeled data and split it

Supervised learning needs pairs: inputs `X` and their correct answers `y`. We synthesize 400 such pairs, then hold back a quarter of them as a **test set**. Keeping test examples separate is what lets us honestly measure how the model does on data it never trained on.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# synthetic labeled data: inputs X, answers y
X, y = make_classification(n_samples=400, n_features=4, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

### Step 2 — Learn a rule from the training pairs

We fit a `LogisticRegression` on only the training pairs. Fitting finds a rule `h(x)` that maps inputs to answers as well as it can on those examples.

In [ ]:
# learn a rule h(x) from the training pairs
model = LogisticRegression().fit(Xtr, ytr)

### Step 3 — Score it on data it never saw

Now we check the rule on both splits. Training accuracy shows how well it fit the examples it learned from; **test** accuracy shows whether that rule generalizes to brand-new inputs — the real goal of supervised learning. We also print a few predictions on unseen examples.

In [ ]:
# answer brand-new inputs it never saw
print("train accuracy:", round(model.score(Xtr, ytr), 3))
print("test  accuracy:", round(model.score(Xte, yte), 3))
print("first 5 predictions:", model.predict(Xte[:5]))

## Visualize the data & results

_Breast-cancer tumors: from 30 cell measurements, can a model tell malignant from benign on new patients?_

### Step 1 — Load, standardize, and split real tumors

We switch to 569 real breast-cancer tumors, each described by 30 cell-nucleus measurements labeled malignant (0) or benign (1). We **standardize** the features so they share a common scale, then split off a test set just like before.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# 569 real tumors, 30 cell-nucleus measurements each
bc = load_breast_cancer()
X = StandardScaler().fit_transform(bc.data)
y = bc.target                      # 0 malignant, 1 benign

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

### Step 2 — Fit a classifier and check test accuracy

We fit logistic regression on the training tumors and measure accuracy on the held-out test tumors. This tells us how well the learned rule diagnoses patients it never saw.

In [ ]:
clf = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
print("test accuracy", clf.score(Xte, yte))

### Step 3 — Project to 2-D and plot the two diagnoses

The inputs live in 30 dimensions, which we can't see directly. **PCA** compresses them down to two dimensions that capture most of the variation, so we can scatter every tumor and watch the malignant and benign groups separate.

In [ ]:
# project the 30-D inputs to 2-D so we can see the two diagnoses
P = PCA(n_components=2, random_state=0).fit_transform(X)

for label, color, name in [(0, "#ff7b72", "malignant"), (1, "#7ee787", "benign")]:
    pts = P[y == label]
    plt.scatter(pts[:, 0], pts[:, 1], c=color, label=name, edgecolor="k")
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.title("Breast-cancer tumors projected to 2-D")
plt.legend()
plt.show()